# Swin-Base Generative AI Attacks on ImageNet-100

PyTorch port of `Model Training/genai-attacks.ipynb`.  
Same attack suite, same structure, adapted for Swin Transformer + ImageNet-100.

**Attacks:**
1. **Inversion Quality Validation** — verify the inverter before running attacks
2. **Latent Interpolation** — interpolate features between classes (α ∈ {0.25, 0.5, 0.75, 1.0})
3. **AdaIN Style Transfer** — spatial AdaIN on Swin token features (fixed 1-D degeneracy)
4. **Prototype Shift** — move features toward class centroids (λ ∈ {0.5, 1.0, 2.0})
5. **Same-Class Control** — shift within same class (null hypothesis)
6. **Diffusion Attack** — noise injection + iterative denoising
7. **Sigma Sensitivity** — vary noise level σ

**Crash recovery:** Per-attack JSON checkpoints in `Swin_Training/genai_results/.checkpoints/`  
**Outputs:** `Swin_Training/genai_results/`

In [ ]:
import sys, os, json, logging, time, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import SwinForImageClassification, AutoImageProcessor
from scipy.stats import entropy as scipy_entropy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR / 'src'))

from swin_utils import (
    seed_everything, get_device, ImageNet100Dataset,
    normalize_imagenet, denormalize_imagenet, total_variation_loss,
    extract_swin_features, extract_swin_spatial_features,
    invert_features_batch_swin, invert_features_swin,
    adain_style_transfer_swin,
    compute_sss_from_confusion,
)

# ── Logging ──
LOG_DIR = PROJECT_ROOT / 'Swin_Training' / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path  = LOG_DIR / f'swin_genai_attacks_{timestamp}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    handlers=[logging.FileHandler(log_path), logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger('swin_genai')
logger.info(f'Log: {log_path}')

In [ ]:
# ── Configuration ──
SEED        = 42
CHECKPOINT  = 'microsoft/swin-base-patch4-window7-224'
NUM_CLASSES = 100
NUM_SAMPLES = 100     # test samples to attack (same as CNN genai notebook)
BATCH_SIZE  = 16

# Inversion hyperparameters
INVERSION_STEPS = 100
INVERSION_LR    = 0.01
TV_WEIGHT       = 1e-4

# Attack parameters
INTERP_ALPHAS = [0.25, 0.5, 0.75, 1.0]
PROTO_LAMBDAS = [0.5, 1.0, 2.0]
DIFFUSION_SIGMAS = [0.05, 0.1, 0.2, 0.4]
DIFFUSION_STEPS  = [5, 10, 20, 50]
SIGMA_SENSITIVITY_SIGMAS = [0.02, 0.05, 0.1, 0.2, 0.3, 0.4]
SIGMA_SENSITIVITY_STEPS  = [5, 10, 20]

DATA_DIR     = PROJECT_ROOT / 'ImageNet100_Training' / 'data'
MAPPING_FILE = PROJECT_ROOT / 'ImageNet100_Training' / 'class_mapping.json'
CKPT_FILE    = PROJECT_ROOT / 'Swin_Training' / 'checkpoints' / 'swin_base_imagenet100_best.pt'

OUT_DIR  = PROJECT_ROOT / 'Swin_Training' / 'genai_results'
CKPT_DIR_SAVE = OUT_DIR / '.checkpoints'
FIG_DIR  = OUT_DIR / 'figures'

for d in [OUT_DIR, CKPT_DIR_SAVE, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device()
logger.info(f'Device: {device}  Samples: {NUM_SAMPLES}')

In [ ]:
# ── Load class mapping ──
with open(MAPPING_FILE) as f:
    class_mapping = json.load(f)
class_names = [
    class_mapping[s]['human_name']
    for s in sorted(class_mapping, key=lambda x: class_mapping[x]['index'])
]

# ── Load test data ──
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
test_ds   = ImageNet100Dataset(DATA_DIR, split='test', processor=processor,
                                max_samples=NUM_SAMPLES)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
logger.info(f'Test subset: {len(test_ds)} samples')

In [ ]:
# ── Load trained Swin model ──
assert CKPT_FILE.exists(), f'Checkpoint not found: {CKPT_FILE}\nRun swin-training.ipynb first.'

model = SwinForImageClassification.from_pretrained(CHECKPOINT)
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
state = torch.load(CKPT_FILE, map_location=device)
model.load_state_dict(state['model'])
model = model.to(device)
model.eval()
logger.info(f'Loaded checkpoint (val_acc={state["val_acc"]:.4f})')

In [ ]:
# ── Precompute test features (GAP + spatial) ──
logger.info('Precomputing test features...')
test_features_gap = {}         # {idx: np.array (1024,)}
test_features_spatial = {}     # {idx: np.array (49, 1024)}
test_images_01 = {}            # {idx: np.array (H, W, 3)} raw [0,1]
test_true_labels = {}          # {idx: int}

idx = 0
for pv_batch, labels_batch in test_loader:
    pv_batch = pv_batch.to(device)
    labels_batch = labels_batch if isinstance(labels_batch, torch.Tensor) else torch.tensor(labels_batch)

    gap  = extract_swin_features(model, pv_batch).cpu().numpy()        # [B, 1024]
    spat = extract_swin_spatial_features(model, pv_batch).cpu().numpy() # [B, 49, 1024]
    imgs = denormalize_imagenet(pv_batch).cpu().numpy()                 # [B, 3, H, W]
    imgs = imgs.transpose(0, 2, 3, 1)                                   # [B, H, W, 3]

    for b in range(pv_batch.size(0)):
        test_features_gap[idx]     = gap[b]
        test_features_spatial[idx] = spat[b]
        test_images_01[idx]        = imgs[b]
        test_true_labels[idx]      = int(labels_batch[b].item())
        idx += 1

logger.info(f'Precomputed features for {idx} samples.')

In [ ]:
# ── Compute class prototypes from training data ──
PROTO_CKPT = CKPT_DIR_SAVE / 'class_prototypes.npy'

if PROTO_CKPT.exists():
    logger.info('Loading class prototypes from checkpoint...')
    class_prototypes = np.load(PROTO_CKPT)  # (NUM_CLASSES, 1024)
else:
    logger.info('Computing class prototypes from training data...')
    train_ds  = ImageNet100Dataset(DATA_DIR, split='train', processor=processor)
    train_dl  = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4)

    proto_sum   = np.zeros((NUM_CLASSES, 1024), dtype=np.float64)
    proto_count = np.zeros(NUM_CLASSES, dtype=np.int64)

    for pv_b, labels_b in train_dl:
        pv_b = pv_b.to(device)
        labels_b = labels_b if isinstance(labels_b, torch.Tensor) else torch.tensor(labels_b)
        feats = extract_swin_features(model, pv_b).cpu().numpy()
        for b in range(pv_b.size(0)):
            lbl = int(labels_b[b].item())
            proto_sum[lbl]   += feats[b]
            proto_count[lbl] += 1

    class_prototypes = (proto_sum / np.maximum(proto_count[:, None], 1)).astype(np.float32)
    np.save(PROTO_CKPT, class_prototypes)
    logger.info(f'Saved class prototypes: {PROTO_CKPT}')

logger.info(f'Class prototypes shape: {class_prototypes.shape}')

In [ ]:
# ── Helper: predict labels from raw [0,1] images ──
@torch.no_grad()
def predict_from_01(images_01, batch_size=BATCH_SIZE):
    """images_01: list of (H,W,3) or np.array (N,H,W,3) in [0,1]. Returns labels, confidences."""
    if isinstance(images_01, list):
        images_01 = np.stack(images_01)
    N = images_01.shape[0]
    preds, confs = [], []
    for start in range(0, N, batch_size):
        batch = images_01[start:start + batch_size]
        t = torch.tensor(batch.transpose(0, 3, 1, 2), dtype=torch.float32, device=device)
        pv = normalize_imagenet(t)
        out = model(pixel_values=pv)
        prob = F.softmax(out.logits, dim=-1)
        conf, pred = prob.max(-1)
        preds.extend(pred.cpu().tolist())
        confs.extend(conf.cpu().tolist())
    return preds, confs

In [ ]:
# ── Attack 0: Inversion Quality Validation ──
CKPT_IQ = CKPT_DIR_SAVE / 'inversion_quality.json'

if CKPT_IQ.exists():
    logger.info('  Checkpoint found — loading inversion_quality.')
    with open(CKPT_IQ) as f:
        inversion_quality = json.load(f)
else:
    logger.info('\n[0] Inversion Quality Validation...')
    IQ_SAMPLES = 20
    inversion_quality = {}

    for i in range(IQ_SAMPLES):
        orig_feat   = test_features_gap[i]         # (1024,)
        orig_img_01 = test_images_01[i]             # (H, W, 3)
        true_lbl    = test_true_labels[i]

        recon = invert_features_swin(
            model, orig_feat, orig_img_01, device,
            steps=INVERSION_STEPS, lr=INVERSION_LR, tv_weight=TV_WEIGHT
        )  # (H, W, 3)

        recon_feat = extract_swin_features(
            model,
            normalize_imagenet(torch.tensor(recon.transpose(2,0,1)[None], dtype=torch.float32, device=device))
        ).cpu().numpy()[0]  # (1024,)

        feat_cos  = float(np.dot(orig_feat, recon_feat) / (np.linalg.norm(orig_feat) * np.linalg.norm(recon_feat) + 1e-8))
        feat_mse  = float(np.mean((orig_feat - recon_feat) ** 2))
        pixel_mse = float(np.mean((orig_img_01 - recon) ** 2))

        inversion_quality[i] = {
            'true_label': true_lbl,
            'true_class': class_names[true_lbl],
            'feat_cosine_sim': round(feat_cos, 4),
            'feat_mse': round(feat_mse, 6),
            'pixel_mse': round(pixel_mse, 6),
        }

    with open(CKPT_IQ, 'w') as f:
        json.dump(inversion_quality, f, indent=2)
    logger.info(f'  Checkpoint saved: {CKPT_IQ.name}')

mean_cos = np.mean([v['feat_cosine_sim'] for v in inversion_quality.values()])
mean_mse = np.mean([v['pixel_mse'] for v in inversion_quality.values()])
logger.info(f'  Inversion quality — mean cosine sim: {mean_cos:.4f}  mean pixel MSE: {mean_mse:.6f}')

In [ ]:
# ── Attack 1: Latent Interpolation ──
# For each sample, interpolate features between true class and a different target class,
# invert each interpolation, and measure if the model flips.
CKPT_LI = CKPT_DIR_SAVE / 'latent_interp_results.json'

if CKPT_LI.exists():
    logger.info('  Checkpoint found — loading latent_interp_results.')
    with open(CKPT_LI) as f:
        latent_interp_results = json.load(f)
else:
    logger.info('\n[1] Latent Interpolation Attack...')
    latent_interp_results = {name: [] for name in ['SwinBase']}

    for i in range(NUM_SAMPLES):
        orig_feat   = test_features_gap[i]
        orig_img_01 = test_images_01[i]
        true_lbl    = test_true_labels[i]

        # Target: pick a different class
        target_lbl  = (true_lbl + 50) % NUM_CLASSES
        target_feat = class_prototypes[target_lbl]

        # Batch all alphas at once
        target_feats = np.array([(1-a)*orig_feat + a*target_feat for a in INTERP_ALPHAS], dtype=np.float32)
        init_imgs    = np.tile(orig_img_01[None], (len(INTERP_ALPHAS), 1, 1, 1))
        recons       = invert_features_batch_swin(
            model, target_feats, init_imgs, device,
            steps=INVERSION_STEPS, lr=INVERSION_LR, tv_weight=TV_WEIGHT
        )  # (len(INTERP_ALPHAS), H, W, 3)

        preds, confs = predict_from_01(recons)

        for ai, alpha in enumerate(INTERP_ALPHAS):
            latent_interp_results['SwinBase'].append({
                'sample_id': i,
                'true_label': true_lbl, 'true_class': class_names[true_lbl],
                'target_label': target_lbl, 'target_class': class_names[target_lbl],
                'alpha': alpha,
                'adv_pred': preds[ai], 'adv_class': class_names[preds[ai]],
                'adv_confidence': round(confs[ai], 4),
                'fooled': preds[ai] != true_lbl,
            })

        if (i + 1) % 10 == 0:
            logger.info(f'  Latent interp: {i+1}/{NUM_SAMPLES}')

    with open(CKPT_LI, 'w') as f:
        json.dump(latent_interp_results, f, indent=2)
    logger.info(f'  Checkpoint saved: {CKPT_LI.name}')

# Fooling rate per alpha
li_records = latent_interp_results['SwinBase']
for alpha in INTERP_ALPHAS:
    alpha_records = [r for r in li_records if r['alpha'] == alpha]
    fr = np.mean([r['fooled'] for r in alpha_records])
    logger.info(f'  Latent Interp α={alpha}: fooling_rate={fr:.3f}')

In [ ]:
# ── Attack 2: AdaIN Style Transfer ──
# Spatial AdaIN on Swin token features (correct version — no 1-D degeneracy).
CKPT_ST = CKPT_DIR_SAVE / 'style_transfer_results.json'

if CKPT_ST.exists():
    logger.info('  Checkpoint found — loading style_transfer_results.')
    with open(CKPT_ST) as f:
        style_transfer_results = json.load(f)
else:
    logger.info('\n[2] AdaIN Style Transfer Attack...')
    style_transfer_results = {'SwinBase': []}

    for i in range(NUM_SAMPLES):
        content_img  = test_images_01[i]
        content_spat = test_features_spatial[i]     # (49, 1024)
        true_lbl     = test_true_labels[i]

        # Style: pick a different class sample
        style_idx    = (i + NUM_SAMPLES // 2) % NUM_SAMPLES
        style_img    = test_images_01[style_idx]
        style_spat   = test_features_spatial[style_idx]  # (49, 1024)

        stylized = adain_style_transfer_swin(
            model, content_img, style_img, device,
            alpha=1.0,
            content_spatial_feat=content_spat,
            style_spatial_feat=style_spat,
            steps=75, lr=INVERSION_LR, tv_weight=TV_WEIGHT,
        )  # (H, W, 3)

        preds, confs = predict_from_01(stylized[None])
        style_transfer_results['SwinBase'].append({
            'sample_id': i,
            'true_label': true_lbl, 'true_class': class_names[true_lbl],
            'style_label': test_true_labels[style_idx],
            'adv_pred': preds[0], 'adv_class': class_names[preds[0]],
            'adv_confidence': round(confs[0], 4),
            'fooled': preds[0] != true_lbl,
        })

        if (i + 1) % 10 == 0:
            logger.info(f'  Style transfer: {i+1}/{NUM_SAMPLES}')

    with open(CKPT_ST, 'w') as f:
        json.dump(style_transfer_results, f, indent=2)
    logger.info(f'  Checkpoint saved: {CKPT_ST.name}')

st_fr = np.mean([r['fooled'] for r in style_transfer_results['SwinBase']])
logger.info(f'  Style Transfer fooling rate: {st_fr:.3f}')

In [ ]:
# ── Attack 3: Prototype Shift ──
# Shift features toward a target class prototype by λ × direction.
CKPT_PS = CKPT_DIR_SAVE / 'proto_shift_results.json'

if CKPT_PS.exists():
    logger.info('  Checkpoint found — loading proto_shift_results.')
    with open(CKPT_PS) as f:
        proto_shift_results = json.load(f)
else:
    logger.info('\n[3] Prototype Shift Attack...')
    proto_shift_results = {'SwinBase': []}

    for i in range(NUM_SAMPLES):
        orig_feat   = test_features_gap[i]
        orig_img_01 = test_images_01[i]
        true_lbl    = test_true_labels[i]

        target_lbl  = (true_lbl + 50) % NUM_CLASSES
        shift_dir   = class_prototypes[target_lbl] - class_prototypes[true_lbl]
        shift_dir   = shift_dir / (np.linalg.norm(shift_dir) + 1e-8)

        # Batch all lambdas
        target_feats = np.array([orig_feat + lam * shift_dir for lam in PROTO_LAMBDAS], dtype=np.float32)
        init_imgs    = np.tile(orig_img_01[None], (len(PROTO_LAMBDAS), 1, 1, 1))
        recons       = invert_features_batch_swin(
            model, target_feats, init_imgs, device,
            steps=INVERSION_STEPS, lr=INVERSION_LR, tv_weight=TV_WEIGHT
        )

        preds, confs = predict_from_01(recons)

        for li, lam in enumerate(PROTO_LAMBDAS):
            proto_shift_results['SwinBase'].append({
                'sample_id': i,
                'true_label': true_lbl, 'true_class': class_names[true_lbl],
                'target_label': target_lbl, 'target_class': class_names[target_lbl],
                'lambda': lam,
                'adv_pred': preds[li], 'adv_class': class_names[preds[li]],
                'adv_confidence': round(confs[li], 4),
                'fooled': preds[li] != true_lbl,
            })

        if (i + 1) % 10 == 0:
            logger.info(f'  Proto shift: {i+1}/{NUM_SAMPLES}')

    with open(CKPT_PS, 'w') as f:
        json.dump(proto_shift_results, f, indent=2)
    logger.info(f'  Checkpoint saved: {CKPT_PS.name}')

for lam in PROTO_LAMBDAS:
    recs = [r for r in proto_shift_results['SwinBase'] if r['lambda'] == lam]
    fr   = np.mean([r['fooled'] for r in recs])
    logger.info(f'  Proto Shift λ={lam}: fooling_rate={fr:.3f}')

In [ ]:
# ── Same-Class Control ──
# Shift toward a SAME-class sample (should NOT fool, validates the proto shift)
CKPT_SC = CKPT_DIR_SAVE / 'proto_shift_control.json'

if CKPT_SC.exists():
    logger.info('  Checkpoint found — loading proto_shift_control.')
    with open(CKPT_SC) as f:
        proto_shift_control = json.load(f)
else:
    logger.info('\n[Control] Same-Class Prototype Shift...')
    proto_shift_control = {'SwinBase': []}

    for i in range(NUM_SAMPLES):
        orig_feat   = test_features_gap[i]
        orig_img_01 = test_images_01[i]
        true_lbl    = test_true_labels[i]

        # Same class prototype → shift direction is within-class
        same_class_proto = class_prototypes[true_lbl]
        shift_dir = same_class_proto - orig_feat
        shift_dir = shift_dir / (np.linalg.norm(shift_dir) + 1e-8)

        target_feats = np.array([orig_feat + lam * shift_dir for lam in PROTO_LAMBDAS], dtype=np.float32)
        init_imgs    = np.tile(orig_img_01[None], (len(PROTO_LAMBDAS), 1, 1, 1))
        recons       = invert_features_batch_swin(
            model, target_feats, init_imgs, device,
            steps=INVERSION_STEPS, lr=INVERSION_LR, tv_weight=TV_WEIGHT
        )

        preds, confs = predict_from_01(recons)

        for li, lam in enumerate(PROTO_LAMBDAS):
            proto_shift_control['SwinBase'].append({
                'sample_id': i,
                'true_label': true_lbl, 'true_class': class_names[true_lbl],
                'lambda': lam,
                'adv_pred': preds[li], 'adv_class': class_names[preds[li]],
                'adv_confidence': round(confs[li], 4),
                'fooled': preds[li] != true_lbl,
            })

        if (i + 1) % 10 == 0:
            logger.info(f'  Same-class control: {i+1}/{NUM_SAMPLES}')

    with open(CKPT_SC, 'w') as f:
        json.dump(proto_shift_control, f, indent=2)
    logger.info(f'  Checkpoint saved: {CKPT_SC.name}')

for lam in PROTO_LAMBDAS:
    recs = [r for r in proto_shift_control['SwinBase'] if r['lambda'] == lam]
    fr   = np.mean([r['fooled'] for r in recs])
    logger.info(f'  Same-Class Control λ={lam}: fooling_rate={fr:.3f} (expected ~0)')

In [ ]:
# ── Attack 4: Diffusion Attack ──
# Add Gaussian noise σ, then iteratively denoise via feature inversion.
CKPT_DI = CKPT_DIR_SAVE / 'diffusion_results.json'

if CKPT_DI.exists():
    logger.info('  Checkpoint found — loading diffusion_results.')
    with open(CKPT_DI) as f:
        _raw = json.load(f)
    diffusion_results = {
        name: {
            'per_step': {int(k): v for k, v in d['per_step'].items()},
            'records': d['records'],
        }
        for name, d in _raw.items()
    }
else:
    logger.info('\n[4] Diffusion Attack...')
    diffusion_results = {
        'SwinBase': {
            'per_step': {s: {'sigma_fooling': {sig: [] for sig in DIFFUSION_SIGMAS}}
                         for s in DIFFUSION_STEPS},
            'records': [],
        }
    }

    for sigma in DIFFUSION_SIGMAS:
        for num_steps in DIFFUSION_STEPS:
            fooled_count = 0
            step_lr = INVERSION_LR / max(num_steps, 1)

            for i in range(NUM_SAMPLES):
                orig_img_01 = test_images_01[i]
                true_lbl    = test_true_labels[i]

                # Add noise
                noisy = (orig_img_01 + np.random.normal(0, sigma, orig_img_01.shape)).clip(0, 1).astype(np.float32)

                # Extract features of noisy image, invert
                noisy_t  = normalize_imagenet(torch.tensor(noisy.transpose(2,0,1)[None], dtype=torch.float32, device=device))
                noisy_feat = extract_swin_features(model, noisy_t).cpu().numpy()[0]  # (1024,)

                denoised = invert_features_swin(
                    model, noisy_feat, noisy, device,
                    steps=num_steps, lr=step_lr, tv_weight=TV_WEIGHT
                )

                preds, confs = predict_from_01(denoised[None])
                fooled = preds[0] != true_lbl
                if fooled:
                    fooled_count += 1

                diffusion_results['SwinBase']['records'].append({
                    'sample_id': i, 'sigma': sigma, 'steps': num_steps,
                    'true_label': true_lbl, 'adv_pred': preds[0],
                    'adv_confidence': round(confs[0], 4), 'fooled': fooled,
                })

            fr = fooled_count / NUM_SAMPLES
            diffusion_results['SwinBase']['per_step'][num_steps]['sigma_fooling'][sigma] = fr
            logger.info(f'  Diffusion σ={sigma} steps={num_steps}: fooling_rate={fr:.3f}')

    with open(CKPT_DI, 'w') as f:
        json.dump({
            name: {'per_step': {str(k): v for k, v in d['per_step'].items()}, 'records': d['records']}
            for name, d in diffusion_results.items()
        }, f, indent=2)
    logger.info(f'  Checkpoint saved: {CKPT_DI.name}')

In [ ]:
# ── Attack 5: Sigma Sensitivity ──
CKPT_SS = CKPT_DIR_SAVE / 'sigma_sensitivity.json'

if CKPT_SS.exists():
    logger.info('  Checkpoint found — loading sigma_sensitivity.')
    with open(CKPT_SS) as f:
        _raw = json.load(f)
    sigma_sensitivity = {
        float(sig_k): {int(step_k): v for step_k, v in step_d.items()}
        for sig_k, step_d in _raw.items()
    }
else:
    logger.info('\n[5] Sigma Sensitivity Analysis...')
    sigma_sensitivity = {}

    for sigma in SIGMA_SENSITIVITY_SIGMAS:
        sigma_sensitivity[sigma] = {}
        for num_steps in SIGMA_SENSITIVITY_STEPS:
            fooled_count = 0
            step_lr = INVERSION_LR / max(num_steps, 1)

            for i in range(NUM_SAMPLES):
                orig_img_01 = test_images_01[i]
                true_lbl    = test_true_labels[i]

                noisy = (orig_img_01 + np.random.normal(0, sigma, orig_img_01.shape)).clip(0, 1).astype(np.float32)
                noisy_t    = normalize_imagenet(torch.tensor(noisy.transpose(2,0,1)[None], dtype=torch.float32, device=device))
                noisy_feat = extract_swin_features(model, noisy_t).cpu().numpy()[0]

                denoised = invert_features_swin(
                    model, noisy_feat, noisy, device,
                    steps=num_steps, lr=step_lr, tv_weight=TV_WEIGHT
                )
                preds, _ = predict_from_01(denoised[None])
                if preds[0] != true_lbl:
                    fooled_count += 1

            fr = fooled_count / NUM_SAMPLES
            sigma_sensitivity[sigma][num_steps] = fr
            logger.info(f'  Sensitivity σ={sigma} steps={num_steps}: fooling={fr:.3f}')

    with open(CKPT_SS, 'w') as f:
        json.dump(
            {str(sig): {str(s): v for s, v in step_d.items()} for sig, step_d in sigma_sensitivity.items()},
            f, indent=2
        )
    logger.info(f'  Checkpoint saved: {CKPT_SS.name}')

In [ ]:
# ── Visualizations ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) Latent interpolation fooling rate vs alpha
ax = axes[0, 0]
li_frs = [np.mean([r['fooled'] for r in li_records if r['alpha'] == a]) for a in INTERP_ALPHAS]
ax.plot(INTERP_ALPHAS, li_frs, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Interpolation α'); ax.set_ylabel('Fooling Rate')
ax.set_title('Latent Interpolation'); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)

# (B) Prototype shift: attack vs control
ax = axes[0, 1]
ps_frs   = [np.mean([r['fooled'] for r in proto_shift_results['SwinBase'] if r['lambda'] == l]) for l in PROTO_LAMBDAS]
ctrl_frs = [np.mean([r['fooled'] for r in proto_shift_control['SwinBase'] if r['lambda'] == l]) for l in PROTO_LAMBDAS]
ax.plot(PROTO_LAMBDAS, ps_frs,   'o-', color='tomato',   label='Proto Shift (attack)', linewidth=2)
ax.plot(PROTO_LAMBDAS, ctrl_frs, 's--', color='gray',    label='Same-Class (control)', linewidth=2)
ax.set_xlabel('Lambda λ'); ax.set_ylabel('Fooling Rate')
ax.set_title('Prototype Shift vs Control'); ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)

# (C) Diffusion step curve (σ=0.1)
ax = axes[1, 0]
sigma_show = 0.1
for sigma_key, step_d in diffusion_results['SwinBase']['per_step'].items():
    frs_by_step = {s: step_d['sigma_fooling'].get(sigma_show, 0) for s in [sigma_key]}
# Plot all steps vs sigma at fixed step count
step_show = DIFFUSION_STEPS[-1]
frs_vs_sigma = [diffusion_results['SwinBase']['per_step'][s]['sigma_fooling'].get(sig, 0)
                for sig in DIFFUSION_SIGMAS
                for s in [step_show]]
# Simpler: one curve per step count
for ns in DIFFUSION_STEPS:
    frs = [diffusion_results['SwinBase']['per_step'][ns]['sigma_fooling'].get(sig, 0)
           for sig in DIFFUSION_SIGMAS]
    ax.plot(DIFFUSION_SIGMAS, frs, 'o-', label=f'{ns} steps', linewidth=2)
ax.set_xlabel('Noise σ'); ax.set_ylabel('Fooling Rate')
ax.set_title('Diffusion Attack'); ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)

# (D) Sigma sensitivity heatmap
ax = axes[1, 1]
heat_data = np.array([[sigma_sensitivity[sig][s] for s in SIGMA_SENSITIVITY_STEPS]
                       for sig in SIGMA_SENSITIVITY_SIGMAS])
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=SIGMA_SENSITIVITY_STEPS,
            yticklabels=SIGMA_SENSITIVITY_SIGMAS, ax=ax)
ax.set_xlabel('Denoising Steps'); ax.set_ylabel('Noise σ')
ax.set_title('Sigma Sensitivity Heatmap')

plt.suptitle('Swin-Base Generative Attack Results', y=1.01, fontsize=13)
plt.tight_layout()
fig.savefig(FIG_DIR / 'genai_attack_summary.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
logger.info('Summary figure saved.')

In [ ]:
# ── Semantic confusion matrix from proto shift ──
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int32)
# Use highest lambda for max signal
max_lam = max(PROTO_LAMBDAS)
for rec in proto_shift_results['SwinBase']:
    if rec['lambda'] == max_lam:
        confusion[rec['true_label'], rec['adv_pred']] += 1

sss_genai = compute_sss_from_confusion(confusion)
logger.info(f'Genai SSS (Proto Shift λ={max_lam}): {sss_genai:.4f}')

# Save confusion matrix
np.save(OUT_DIR / 'SwinBase_semantic_confusion_matrix.npy', confusion)
pd.DataFrame(confusion, index=class_names[:NUM_CLASSES],
             columns=class_names[:NUM_CLASSES]).to_csv(
    OUT_DIR / 'SwinBase_confusion_matrix.csv'
)

# Top confused pairs
pairs = []
for ti in range(NUM_CLASSES):
    for aj in range(NUM_CLASSES):
        if ti != aj and confusion[ti, aj] > 0:
            pairs.append({'true_class': class_names[ti], 'adv_class': class_names[aj],
                          'count': int(confusion[ti, aj])})
pairs.sort(key=lambda x: x['count'], reverse=True)
print('\nTop-5 confused pairs (Proto Shift):')
for p in pairs[:5]:
    print(f'  {p["true_class"]} → {p["adv_class"]} (n={p["count"]})')

In [ ]:
# ── Save combined results ──
genai_summary = {
    'model': 'SwinBase', 'dataset': 'ImageNet-100',
    'num_samples': NUM_SAMPLES,
    'inversion_quality': {
        'mean_cosine_sim': float(np.mean([v['feat_cosine_sim'] for v in inversion_quality.values()])),
        'mean_pixel_mse':  float(np.mean([v['pixel_mse']       for v in inversion_quality.values()])),
    },
    'latent_interp': {
        str(a): float(np.mean([r['fooled'] for r in li_records if r['alpha'] == a]))
        for a in INTERP_ALPHAS
    },
    'style_transfer': {'fooling_rate': float(st_fr)},
    'proto_shift': {
        str(l): float(np.mean([r['fooled'] for r in proto_shift_results['SwinBase'] if r['lambda'] == l]))
        for l in PROTO_LAMBDAS
    },
    'same_class_control': {
        str(l): float(np.mean([r['fooled'] for r in proto_shift_control['SwinBase'] if r['lambda'] == l]))
        for l in PROTO_LAMBDAS
    },
    'sss_proto_shift': sss_genai,
    'timestamp': datetime.now().isoformat(),
}

with open(OUT_DIR / 'genai_results.json', 'w') as f:
    json.dump(genai_summary, f, indent=2)

pd.DataFrame([genai_summary]).to_csv(OUT_DIR / 'genai_summary.csv', index=False)
logger.info('All results saved.')

In [ ]:
# ── Summary ──
print('=' * 60)
print('  SWIN-BASE GENERATIVE ATTACKS SUMMARY')
print('=' * 60)
print(f'  Inversion quality — cosine sim: {genai_summary["inversion_quality"]["mean_cosine_sim"]:.4f}')
print(f'\n  Latent Interp fooling rates:')
for a, fr in genai_summary['latent_interp'].items():
    print(f'    α={a}: {fr:.3f}')
print(f'\n  Style Transfer fooling rate: {genai_summary["style_transfer"]["fooling_rate"]:.3f}')
print(f'\n  Proto Shift fooling rates:')
for l, fr in genai_summary['proto_shift'].items():
    print(f'    λ={l}: {fr:.3f}')
print(f'\n  Same-Class Control (should be ~0):')
for l, fr in genai_summary['same_class_control'].items():
    print(f'    λ={l}: {fr:.3f}')
print(f'\n  Genai SSS (Proto Shift): {sss_genai:.4f}')
print(f'\n  Log: {log_path}')
print('=' * 60)